# BKT Model — Training & Performance Analysis

This notebook trains the Bayesian Knowledge Tracing model and produces publication-ready figures exportable to PDF.

**Sections**
1. Setup & Data Loading
2. Model Training with Parameter History
3. EM Convergence — 4 paramètres par compétence (grille 6 compétences)
4. **EM Convergence — 1 graphe par paramètre, toutes compétences**
5. Distribution finale des paramètres
6. Trajectoires de connaissance
7. Courbes d'apprentissage
8. Courbe de calibration
9. Courbe ROC & métriques globales
10. Cross-Validation
11. Performance par compétence
12. Export PDF

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, log_loss
from sklearn.calibration import calibration_curve

from bkt_model import my_bkt_vectorized
from bkt_launcher import get_default_parameters, cross_validate_bkt, create_student_folds_per_skill

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
ALL_FIGS = []   # every figure appended here → written to PDF at the end
print('Setup OK')

## 1. Data Loading

In [ ]:
DATA_PATH = '../../datas_management/cleaned_events.csv'

df = pd.read_csv(DATA_PATH)

mapping = {
    'student_id': 'id_eleve',
    'timestamp':  'timestamp',
    'skill_id':   'id_competence',
    'result':     'bonne_reponse',
}

df = df.sort_values([mapping['student_id'], mapping['skill_id'], mapping['timestamp']]).reset_index(drop=True)

print(f"Observations : {len(df):,}")
print(f"Etudiants    : {df[mapping['student_id']].nunique():,}")
print(f"Compétences  : {df[mapping['skill_id']].nunique():,}")
print(f"Taux de réussite global : {df[mapping['result']].mean():.2%}")
df.head(3)

## 2. Model Training with Parameter History Tracking

We subclass `my_bkt_vectorized` to record parameters after every EM iteration.

In [ ]:
class BKTWithHistory(my_bkt_vectorized):
    """BKT model that records parameter history across EM iterations."""

    def em_algorithm(self, iterations=100, convergence_threshold=0):
        self.param_history = {skill: [self.parameters_dict[skill].copy()]
                               for skill in self.parameters_dict}
        self.n_iters_run = 0

        for iteration in range(iterations):
            l_dict = self.e_step()
            new_params = self.m_step(l_dict)

            converged = True
            for skill in new_params:
                if np.max(np.abs(self.parameters_dict[skill] - new_params[skill])) > convergence_threshold:
                    converged = False
                self.parameters_dict[skill] = new_params[skill]
                self.param_history[skill].append(new_params[skill].copy())

            self.n_iters_run = iteration + 1
            if converged and iteration > 5:
                print(f'Convergence atteinte à l\'itération {iteration}')
                break

        for skill in self.param_history:
            self.param_history[skill] = np.array(self.param_history[skill])

In [ ]:
initial_parameters = get_default_parameters(df, mapping['skill_id'])

bkt = BKTWithHistory(df, df, mapping, initial_parameters)
bkt.em_algorithm(iterations=100)

print(f'\nIterations exécutées : {bkt.n_iters_run}')
print(f'Compétences entraînées : {len(bkt.param_history)}')

## 3. EM Convergence — Parameter Evolution per Skill

In [ ]:
param_names  = ['p_0 (Initial)', 'p_T (Learn)', 'p_G (Guess)', 'p_S (Slip)']
param_colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
skills       = list(bkt.param_history.keys())

# Show up to 6 skills in a 2x3 grid
n_show = min(6, len(skills))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, skill in zip(axes[:n_show], skills[:n_show]):
    history = bkt.param_history[skill]  # shape (n_iters, 4)
    iters   = np.arange(len(history))
    for i, (name, color) in enumerate(zip(param_names, param_colors)):
        ax.plot(iters, history[:, i], label=name, color=color, linewidth=1.8)
    ax.set_title(str(skill), fontsize=10)
    ax.set_xlabel('Itération EM')
    ax.set_ylabel('Valeur du paramètre')
    ax.set_ylim(0, 1)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.02))

# hide unused axes
for ax in axes[n_show:]:
    ax.set_visible(False)

fig.suptitle('Évolution des paramètres BKT au cours des itérations EM', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/01_em_convergence.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 4. EM Convergence — 1 graphe par paramètre, toutes compétences

Chaque sous-graphe montre l'évolution d'**un seul paramètre** au fil des itérations EM pour **toutes les compétences simultanément** (lignes fines) ainsi que la moyenne ± 1 écart-type (ligne épaisse + zone ombrée).

In [ ]:
n_iters_max = max(len(bkt.param_history[s]) for s in skills)

# Align all skill histories to the same length by padding with last value
param_arrays = []
for s in skills:
    h = bkt.param_history[s]          # shape (ni, 4)
    ni = len(h)
    if ni < n_iters_max:
        pad = np.tile(h[-1], (n_iters_max - ni, 1))
        h = np.vstack([h, pad])
    param_arrays.append(h)

param_tensor = np.stack(param_arrays)  # (n_skills, n_iters_max, 4)
iters = np.arange(n_iters_max)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (ax, name, color) in enumerate(zip(axes, param_names, param_colors)):
    # Individual skill traces
    for s_idx in range(len(skills)):
        ax.plot(iters, param_tensor[s_idx, :, idx],
                color=color, alpha=0.20, linewidth=0.9)

    # Mean ± std across skills
    mean_curve = param_tensor[:, :, idx].mean(axis=0)
    std_curve  = param_tensor[:, :, idx].std(axis=0)

    ax.fill_between(iters,
                    np.clip(mean_curve - std_curve, 0, 1),
                    np.clip(mean_curve + std_curve, 0, 1),
                    alpha=0.18, color='black', label='± 1 std')
    ax.plot(iters, mean_curve, color='black', linewidth=2.5, label='Moyenne')

    # Annotate final mean value
    ax.annotate(f'{mean_curve[-1]:.3f}',
                xy=(iters[-1], mean_curve[-1]),
                xytext=(8, 0), textcoords='offset points',
                fontsize=9, va='center', color='black')

    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Itération EM')
    ax.set_ylabel('Valeur du paramètre')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right')

fig.suptitle(
    f'Convergence EM — évolution de chaque paramètre BKT\n'
    f'({len(skills)} compétences · lignes fines = individuel · ligne épaisse = moyenne)',
    fontsize=13
)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/01b_params_per_epoch_all_skills.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 4. Final Parameter Distributions across Skills

In [ ]:
# Collect final parameters for all skills
final_params = np.array([bkt.parameters_dict[s] for s in skills])
# final_params shape: (n_skills, 4)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (ax, name, color) in enumerate(zip(axes, param_names, param_colors)):
    values = final_params[:, i]
    ax.hist(values, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(values.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Moy. {values.mean():.3f}')
    ax.set_title(name)
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Nombre de compétences' if i == 0 else '')
    ax.legend()

fig.suptitle('Distribution des paramètres BKT finaux (toutes compétences)', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/02_parameter_distributions.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

params_df = pd.DataFrame(final_params, index=skills, columns=['p_0', 'p_T', 'p_G', 'p_S'])
params_df.describe().round(4)

## 5. Knowledge State Trajectories

Forward-pass knowledge estimates for sampled students on a selected skill.

In [ ]:
def plot_knowledge_trajectories(bkt, skill, n_students=8, ax=None):
    """Plot P(knowledge) over time for n_students on a given skill."""
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    students = list(bkt.grouped_data[skill].keys())[:n_students]
    cmap = plt.cm.viridis(np.linspace(0, 0.9, len(students)))

    for student, color in zip(students, cmap):
        results = bkt.grouped_data[skill][student]
        l_fwd, _ = bkt.forward_pass_vectorized(results, skill)
        t = np.arange(len(l_fwd))
        ax.plot(t, l_fwd, color=color, linewidth=1.5, alpha=0.8)
        for i, r in enumerate(results):
            marker = '^' if r == 1 else 'v'
            mcolor = '#4CAF50' if r == 1 else '#E91E63'
            ax.scatter(i + 1, l_fwd[i + 1], marker=marker, color=mcolor, s=40, zorder=5, alpha=0.7)

    ax.axhline(0.95, color='gray', linestyle=':', linewidth=1, label='Seuil maîtrise 0.95')
    ax.set_xlabel('Tentative')
    ax.set_ylabel('P(Connaissance)')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'Trajectoires de connaissance — {skill}')
    ax.legend()
    if show:
        plt.tight_layout()
        return ax.figure


# Pick up to 3 skills with most students
skill_student_counts = {s: len(bkt.grouped_data[s]) for s in skills}
top_skills = sorted(skill_student_counts, key=skill_student_counts.get, reverse=True)[:3]

fig, axes = plt.subplots(1, len(top_skills), figsize=(16, 5))
if len(top_skills) == 1:
    axes = [axes]

for ax, skill in zip(axes, top_skills):
    plot_knowledge_trajectories(bkt, skill, n_students=10, ax=ax)

fig.suptitle('Évolution de P(Connaissance) par étudiant (▲ correct, ▼ incorrect)', fontsize=13)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/03_knowledge_trajectories.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 6. Learning Curves — Accuracy vs. Attempt Number

In [ ]:
# For each attempt index, compute observed accuracy and mean predicted P(correct)
preds_df = bkt.predict_with_results()

# Add attempt index per student-skill pair
preds_df['attempt'] = preds_df.groupby([mapping['student_id'], mapping['skill_id']]).cumcount() + 1

MAX_ATTEMPT = 15
curve = (
    preds_df[preds_df['attempt'] <= MAX_ATTEMPT]
    .groupby('attempt')
    .agg(
        obs_accuracy=('actual_result', 'mean'),
        pred_probability=('predicted_result', 'mean'),
        count=('actual_result', 'size'),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curve['attempt'], curve['obs_accuracy'],   'o-', color='#2196F3', linewidth=2, label='Taux de réussite observé')
ax.plot(curve['attempt'], curve['pred_probability'], 's--', color='#FF9800', linewidth=2, label='Probabilité prédite moyenne')

ax2 = ax.twinx()
ax2.bar(curve['attempt'], curve['count'], alpha=0.15, color='gray', label='Nb observations')
ax2.set_ylabel('Nombre d\'observations', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')
ax2.spines['top'].set_visible(False)

ax.set_xlabel('Numéro de tentative')
ax.set_ylabel('Probabilité / Taux de réussite')
ax.set_title('Courbe d\'apprentissage — Taux de réussite observé vs. probabilité prédite')
ax.set_ylim(0, 1)
ax.set_xticks(curve['attempt'])

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/04_learning_curves.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 7. Prediction Calibration Curve

In [ ]:
y_true = preds_df['actual_result'].values.astype(float)
y_pred = preds_df['predicted_result'].values.astype(float)

fraction_pos, mean_pred = calibration_curve(y_true, y_pred, n_bins=15, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Calibration plot
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Calibration parfaite')
ax.plot(mean_pred, fraction_pos, 'o-', color='#2196F3', linewidth=2, markersize=6, label='BKT')
ax.fill_between(mean_pred, fraction_pos, mean_pred,
                where=(fraction_pos > mean_pred), alpha=0.1, color='#4CAF50', label='Sur-confiance')
ax.fill_between(mean_pred, fraction_pos, mean_pred,
                where=(fraction_pos < mean_pred), alpha=0.1, color='#E91E63', label='Sous-confiance')
ax.set_xlabel('Probabilité prédite moyenne')
ax.set_ylabel('Fraction de positifs observés')
ax.set_title('Courbe de calibration')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# Prediction histogram
ax = axes[1]
ax.hist(y_pred[y_true == 1], bins=40, alpha=0.6, color='#4CAF50', label='Réponses correctes', density=True)
ax.hist(y_pred[y_true == 0], bins=40, alpha=0.6, color='#E91E63', label='Réponses incorrectes', density=True)
ax.set_xlabel('Probabilité prédite P(correct)')
ax.set_ylabel('Densité')
ax.set_title('Distribution des prédictions par résultat réel')
ax.legend()

fig.suptitle('Analyse de la calibration du modèle BKT', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/05_calibration.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 8. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_pred)
auc_val      = roc_auc_score(y_true, y_pred)
rmse_val     = np.sqrt(mean_squared_error(y_true, y_pred))
logloss_val  = log_loss(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
ax = axes[0]
ax.plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'BKT (AUC = {auc_val:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aléatoire (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbe ROC')
ax.legend()

# Metric summary bar chart
ax = axes[1]
metrics = {'AUC': auc_val, 'RMSE': rmse_val, 'Log Loss': logloss_val}
colors  = ['#4CAF50', '#FF9800', '#E91E63']
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, max(metrics.values()) * 1.25)
ax.set_title('Métriques globales')
ax.set_ylabel('Valeur')

fig.suptitle('Performance globale du modèle BKT', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/06_roc_metrics.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 9. Cross-Validation Results

In [ ]:
CV_PATH = 'cv_results2.csv'
cv_df = pd.read_csv(CV_PATH)
print(cv_df)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metric_cols   = ['auc', 'rmse', 'log_v']
metric_labels = ['AUC', 'RMSE', 'Log Loss']
metric_colors = ['#4CAF50', '#FF9800', '#E91E63']

for ax, col, label, color in zip(axes, metric_cols, metric_labels, metric_colors):
    values = cv_df[col].values
    folds  = cv_df['fold'].values

    ax.bar(folds, values, color=color, edgecolor='white', alpha=0.85, width=0.6)
    ax.axhline(values.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Moy. {values.mean():.4f} ± {values.std():.4f}')
    ax.fill_between([0.5, len(folds) + 0.5],
                    values.mean() - values.std(),
                    values.mean() + values.std(),
                    alpha=0.12, color='black')

    for i, (fold, v) in enumerate(zip(folds, values)):
        ax.text(fold, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

    ax.set_title(label)
    ax.set_xlabel('Fold')
    ax.set_ylabel(label)
    ax.set_xticks(folds)
    ax.legend(fontsize=9)

fig.suptitle('Résultats de la Cross-Validation 5-Folds', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/07_cross_validation.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 10. Per-Skill Performance

In [ ]:
# Compute AUC per skill (only skills with at least 2 classes)
per_skill_rows = []
for skill in preds_df[mapping['skill_id']].unique():
    sdf  = preds_df[preds_df[mapping['skill_id']] == skill]
    yt   = sdf['actual_result'].values.astype(float)
    yp   = sdf['predicted_result'].values.astype(float)
    if len(np.unique(yt)) < 2:
        continue
    auc_s  = roc_auc_score(yt, yp)
    rmse_s = np.sqrt(mean_squared_error(yt, yp))
    p0, pt, pg, ps = bkt.parameters_dict[skill]
    per_skill_rows.append({
        'skill': skill, 'auc': auc_s, 'rmse': rmse_s,
        'p_0': p0, 'p_T': pt, 'p_G': pg, 'p_S': ps,
        'n_obs': len(sdf),
    })

skill_perf = pd.DataFrame(per_skill_rows).sort_values('auc', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# AUC per skill (horizontal bar)
ax = axes[0]
colors = plt.cm.RdYlGn(skill_perf['auc'].values)
bars = ax.barh(skill_perf['skill'], skill_perf['auc'], color=colors, edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='Aléatoire')
ax.axvline(skill_perf['auc'].mean(), color='black', linestyle='--', linewidth=1.2,
           label=f'Moy. {skill_perf["auc"].mean():.3f}')
ax.set_xlabel('AUC')
ax.set_title('AUC par compétence')
ax.set_xlim(0, 1)
ax.legend()

# Scatter: p_T (learn rate) vs AUC, sized by n_obs
ax = axes[1]
sc = ax.scatter(
    skill_perf['p_T'], skill_perf['auc'],
    c=skill_perf['p_0'], cmap='coolwarm',
    s=skill_perf['n_obs'] / skill_perf['n_obs'].max() * 300 + 30,
    alpha=0.8, edgecolors='white', linewidth=0.5
)
plt.colorbar(sc, ax=ax, label='p_0 (connaissance initiale)')
ax.set_xlabel('p_T (taux d\'apprentissage)')
ax.set_ylabel('AUC')
ax.set_title('AUC vs. Taux d\'apprentissage (taille = nb. observations, couleur = p_0)')

fig.suptitle('Performance par compétence', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/08_per_skill_performance.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

print(skill_perf[['skill', 'auc', 'rmse', 'p_0', 'p_T', 'p_G', 'p_S', 'n_obs']].to_string(index=False))

## 11. Export all figures to a single PDF

In [ ]:
output_pdf = f'{FIGURES_DIR}/bkt_analysis_report.pdf'

if ALL_FIGS:
    with PdfPages(output_pdf) as pdf:
        for fig in ALL_FIGS:
            pdf.savefig(fig, bbox_inches='tight')
    print(f'Rapport PDF exporté : {output_pdf}')
    print(f'Figures incluses    : {len(ALL_FIGS)}')
else:
    print('Aucune figure enregistrée.')
    print('Relancez toutes les cellules depuis le début, puis ré-exécutez cette cellule.')
    print()
    print('Alternative sans re-run :')
    print('  jupyter nbconvert --to pdf bkt_analysis.ipynb')

> **Tip — export sans PyMuPDF** : si PyMuPDF n'est pas installé, utilisez `jupyter nbconvert --to pdf bkt_analysis.ipynb` depuis le terminal pour exporter tout le notebook (cellules + figures) en un seul PDF.